# The Corpus We Cannot Ship

**Computational Text Analysis Pack · Module 13 — Case Studies**

*Technique:* Copyright, redistribution and reproducibility as method constraints

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/16-case-studies/13c_the_corpus_we_cannot_ship.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

The 61 novels in this case study are not in this course, and will not be. Every
one is in copyright — Achebe, Gordimer, Ngũgĩ, Okri, Emecheta, Forna, Tutuola —
and the files carry the fingerprints of a shadow library in their names.

That is not a footnote. It determines what the project can claim, who can check
it, and whether it is research at all in the sense that matters: **reproducible
by someone who is not you.**

## The three separate problems

They get conflated, and they have different remedies.

1. **Legality of *use*.** Text and data mining exceptions differ by jurisdiction.
   The EU's CDSM Directive Arts. 3–4 permit TDM for research; the UK exception
   covers non-commercial research. Reading in-copyright text *computationally* is
   often lawful where redistributing it never is.
2. **Legality of *redistribution*.** Almost always no. This is why the course
   ships public-domain and openly-licensed corpora only, and why
   `fetch_text_corpora.py` has an `assert_clean()` that fails the build if a
   quarantined path appears.
3. **Provenance.** Where did the file come from? A corpus assembled from a
   shadow library cannot be cited, cannot be verified, and may not be the text
   you think — editions differ, OCR differs, and you have no way to check.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
# What the course ships, and why each is safe.
shipped = [
    ("colonial_texts.csv", "Project Gutenberg", "Public domain",
     "pre-1929, verifiable edition, citable"),
    ("masakhanews.csv", "Masakhane / MasakhaNEWS", "AFL-3.0",
     "openly licensed, versioned, benchmark"),
    ("masakhaner.csv", "Masakhane / MasakhaNER 2.0", "CC BY 4.0",
     "openly licensed, official splits"),
    ("african_legal.csv", "Internet Archive / open government", "Public / open",
     "institutional publication"),
]
print(f"{'file':<22}{'source':<30}{'licence':<16}why it is shippable")
print("-" * 96)
for f, s, l, w in shipped:
    print(f"{f:<22}{s:<30}{l:<16}{w}")
print()
print("The 61 novels satisfy none of these columns.")

## What you can still do with material you cannot share

This is the practical part, and the answer is: quite a lot.

- **Publish the code, not the corpus.** The pipeline, the parameters and the
  derived measurements are yours to release.
- **Publish derived data.** Word frequencies, document-term matrices and model
  weights are generally distributable where the text is not — this is the
  "non-consumptive research" principle that governs HathiTrust's Extracted
  Features datasets.
- **Point at an acquirable edition.** Cite ISBNs. A reader who buys the same
  edition can rerun you.
- **Demonstrate on a clean corpus.** Which is exactly what the Lab Notebook in
  13d does: the same pipeline, on text this course can hand you.

In [ ]:
# Derived data is shippable where text is not: a frequency profile carries the
# signal an attribution study needs, and cannot reconstruct the book.
works = (df_colonial.groupby(["work_id", "author"])["text"]
         .apply(" ".join).reset_index())
sample = works.iloc[0]
toks = tokenise(sample["text"])
prof = Counter(toks)
top = prof.most_common(30)

print(f"work {sample['work_id']} by {sample['author']}")
print(f"   {len(toks):,} tokens -> a 30-number profile\n")
print("   " + ", ".join(f"{w}:{n}" for w, n in top[:12]))
print()
print(f"reconstruct the text from that? No. Run Burrows's Delta on it? Yes —")
print(f"module 9 needs nothing else. Function-word rates ARE the derived data")
print(f"that makes in-copyright stylometry publishable.")

### 🔧 Your turn 1

Take a corpus you cannot redistribute and write the three sentences you would put in a data-availability statement for it.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

A usable pattern: name the works and editions precisely enough that a reader can
acquire them; state which derived data you are depositing and where (a DOI on
Zenodo or OSF); and state the legal basis for your own use. The commonest
failure is silence — a paper that reports results on "a corpus of 61 novels"
with no account of provenance is not reproducible, and reviewers increasingly
say so. Note this is the same demand module 1's "Whose Africa Is in the Public
Domain?" made about corpus construction, arriving from the legal side instead of
the political one.

</details>